# Validation orqali WER va CER ko‘rsatkichlarini yaxshilash

Fayl: `07_optimize_validation_wer_cer.ipynb`. Ishga tushirish va papkalar izohi: [README.md](README.md).


In [ ]:
from pathlib import Path
import io, json, math, random, hashlib, unicodedata
from collections import Counter, OrderedDict
from itertools import product
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer, process_words
from tqdm.auto import tqdm

PROJECT = Path.cwd().resolve()
assert (PROJECT / "data/train_metadata.parquet").exists(), "Loyiha ildizidan oching."
CHECKPOINT = PROJECT / "checkpoints/full_lora_step_8500"
BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
OUT = PROJECT / "results/stt_wer_cer_v2"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
BUILD_EXTRA_LM = False
DO_FINETUNE = False
MAX_EPOCHS = 6  # Qo‘shimcha fine-tuning uchun maksimal epoch soni.
RUN_FINAL_TEST = False  # Faqat validation tanlovi tugagandan keyin True.

In [ ]:
def normalize_text(value):
    if not isinstance(value, str):
        raise ValueError("Matn string bo‘lishi kerak; NaN ni 'nan'ga aylantirmang.")
    return " ".join(unicodedata.normalize("NFC", value).lower().split())

processor = Wav2Vec2Processor.from_pretrained(
    str(CHECKPOINT), local_files_only=True
)
tokenizer = processor.tokenizer
vocab = tokenizer.get_vocab()
blank_id = tokenizer.pad_token_id
allowed_chars = {t for t in vocab if len(t) == 1 and t != "|"}
assert set(vocab.values()) == set(range(len(vocab)))
assert blank_id is not None and tokenizer.word_delimiter_token == "|"

labels = [tokenizer.convert_ids_to_tokens(i) for i in range(len(vocab))]
labels[blank_id] = ""  # CTC blank aynan PAD ustunida.
labels[vocab["|"]] = " "
assert len(set(labels)) == len(labels), "Decoder label takrorlangan."

In [ ]:
frames = {}
for split in ("train", "val", "test"):
    df = pd.read_parquet(PROJECT / f"data/{split}_metadata.parquet")
    required = {"id", "cleaned_text_lowercase", "source_file", "source_row"}
    assert required <= set(df.columns), f"{split}: ustunlar yetishmaydi."
    assert df[list(required - {"cleaned_text_lowercase"})].notna().all().all()
    rows = pd.to_numeric(df["source_row"], errors="raise")
    assert ((rows >= 0) & (rows == np.floor(rows))).all()
    df["source_row"] = rows.astype(int)
    df["text"] = df["cleaned_text_lowercase"].map(
        lambda t: normalize_text(t) if isinstance(t, str) else ""
    )
    df["bad_text"] = df["text"].map(
        lambda t: not t or any(c not in allowed_chars and c != " " for c in t)
    )
    assert not df["id"].duplicated().any(), f"{split}: takror ID."
    if split != "train":
        assert not df["bad_text"].any(), f"{split}: reference matnini tekshiring."
        assert not df.duplicated(["source_file", "source_row"]).any()
    frames[split] = df

for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (set(frames[left]["id"]) & set(frames[right]["id"])), "Split ID leakage."
    loc = lambda d: set(zip(d["source_file"], d["source_row"]))
    assert not (loc(frames[left]) & loc(frames[right])), "Split audio leakage."

train_df = frames["train"].copy()
heldout_texts = set(frames["val"]["text"]) | set(frames["test"]["text"])
remove = (train_df["bad_text"] | train_df.duplicated(["source_file", "source_row"])
          | train_df["text"].isin(heldout_texts))
train_df.loc[remove].to_csv(OUT / "train_quarantine.csv", index=False)
train_df = train_df.loc[~remove].reset_index(drop=True)
val_df = frames["val"].reset_index(drop=True)
test_df = frames["test"].reset_index(drop=True)
assert len(train_df) and len(val_df) and len(test_df)
train_df.to_parquet(OUT / "train_clean_metadata.parquet", index=False)

# Audio fayl mavjudligi va source_row chegarasi — audio decoding qilmasdan.
all_meta = pd.concat(list(frames.values()), ignore_index=True)
for name, group in all_meta.groupby("source_file", sort=False):
    shard = PROJECT / "audio_shards" / name
    assert shard.is_file(), f"Shard yo‘q: {shard}"
    count = pq.ParquetFile(shard).metadata.num_rows
    assert group["source_row"].max() < count, f"source_row noto‘g‘ri: {name}"
print({"train": len(train_df), "val": len(val_df), "test": len(test_df),
       "train_quarantine": int(remove.sum())})

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, metadata, cache_size=4):
        self.df = metadata.reset_index(drop=True)
        self.files, self.ends = {}, {}
        self.cache = OrderedDict()
        self.cache_size = cache_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[int(index)]
        name, position = row["source_file"], int(row["source_row"])
        if name not in self.files:
            self.files[name] = pq.ParquetFile(PROJECT / "audio_shards" / name)
            meta = self.files[name].metadata
            self.ends[name] = np.cumsum([
                meta.row_group(i).num_rows for i in range(meta.num_row_groups)
            ])
        ends = self.ends[name]
        assert 0 <= position < ends[-1]
        group = int(np.searchsorted(ends, position, side="right"))
        local = position - (int(ends[group - 1]) if group else 0)
        key = (name, group)
        if key not in self.cache:
            self.cache[key] = self.files[name].read_row_group(
                group, columns=["id", "audio"]
            ).to_pylist()
            if len(self.cache) > self.cache_size:
                self.cache.popitem(last=False)
        self.cache.move_to_end(key)
        record = self.cache[key][local]
        assert str(record["id"]) == str(row["id"]), "Audio/reference ID mos emas."
        payload = record["audio"]
        if isinstance(payload, dict):
            payload = payload.get("bytes")
        if not isinstance(payload, (bytes, bytearray)):
            raise ValueError(f"Audio bytes yo‘q: {row['id']}")
        audio, sr = sf.read(io.BytesIO(payload), dtype="float32")
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if not len(audio) or not np.isfinite(audio).all() or sr <= 0:
            raise ValueError(f"Audio buzilgan: {row['id']}")
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        if not np.any(audio):
            raise ValueError(f"Audio to‘liq jim: {row['id']}")
        return {"id": row["id"], "audio": np.asarray(audio, dtype=np.float32),
                "text": row["text"]}

train_data = AudioDataset(train_df)
val_data = AudioDataset(val_df)
test_data = AudioDataset(test_df)

In [ ]:
checkpoint_path = CHECKPOINT / "training_state.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert int(checkpoint["global_step"]) == 8500
assert checkpoint["vocab_size"] == len(vocab)
assert checkpoint["base_model_name"] == "facebook/wav2vec2-xls-r-300m"
base = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL, local_files_only=True,
    vocab_size=len(vocab), pad_token_id=blank_id,
    ignore_mismatched_sizes=True,
    ctc_loss_reduction="mean", ctc_zero_infinity=False
)
for parameter in base.parameters():
    parameter.requires_grad = False
lora = LoraConfig.from_pretrained(str(CHECKPOINT))
lora.inference_mode = False
model = get_peft_model(base, lora)
weights = checkpoint["model_weights"]
current = model.state_dict()
required_weights = {name for name, p in model.named_parameters() if p.requires_grad}
assert required_weights <= set(weights), "LoRA yoki CTC head og‘irliklari yetishmaydi."
assert any("lm_head" in name for name in required_weights)
for name, value in weights.items():
    assert name in current and current[name].shape == value.shape, name
loaded = model.load_state_dict(weights, strict=False)
assert not loaded.unexpected_keys
assert not (required_weights & set(loaded.missing_keys))
model.to(DEVICE).eval()
model_tag = "step8500"
del checkpoint, weights, current, base

In [ ]:
def collate_ctc(samples):
    batch = processor([s["audio"] for s in samples], sampling_rate=16000,
                      padding=True, return_attention_mask=True, return_tensors="pt")
    targets = tokenizer([s["text"] for s in samples], padding=True,
                        add_special_tokens=False, return_tensors="pt")
    labels_ctc = targets.input_ids.masked_fill(targets.attention_mask == 0, -100)
    assert not ((labels_ctc == blank_id) | (labels_ctc == tokenizer.unk_token_id)).any()
    lengths = model.get_base_model()._get_feat_extract_output_lengths(
        batch.attention_mask.sum(-1)
    )
    for ids, length in zip(labels_ctc, lengths):
        ids = ids[ids != -100]
        minimum = len(ids) + int((ids[1:] == ids[:-1]).sum())
        if minimum > int(length):
            raise ValueError("CTC alignment imkonsiz: audio/transcriptni tekshiring.")
    return {**dict(batch), "labels": labels_ctc}

@torch.inference_mode()
def get_logits(sample):
    model.eval()
    inputs = processor(sample["audio"], sampling_rate=16000,
                       return_attention_mask=True, return_tensors="pt")
    logits = model(**{k: v.to(DEVICE) for k, v in inputs.items()}).logits[0]
    result = logits.float().cpu().numpy()
    assert result.ndim == 2 and result.shape[1] == len(vocab)
    assert np.isfinite(result).all()
    return result

def greedy_validation():
    refs, hyps = [], []
    for i in tqdm(range(len(val_data)), desc="Acoustic validation"):
        sample = val_data[i]
        refs.append(sample["text"])
        hyps.append(normalize_text(processor.decode(get_logits(sample).argmax(-1))))
    return {"wer": wer(refs, hyps) * 100, "cer": cer(refs, hyps) * 100}

In [ ]:
if DO_FINETUNE:
    loader = DataLoader(train_data, batch_size=1, shuffle=True,
                        collate_fn=collate_ctc, num_workers=0)
    trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": [p for name, p in trainable if "lm_head" not in name], "lr": 1e-5},
        {"params": [p for name, p in trainable if "lm_head" in name], "lr": 3e-5}
    ], weight_decay=0.01)
    best_metric = greedy_validation()
    best_state = {name: p.detach().cpu().clone() for name, p in trainable}
    improved, history = False, []
    accumulation = 4
    for epoch in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for index, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch + 1}")):
            # Oxirgi kichik accumulation guruhi ham to‘g‘ri normalizatsiya qilinadi.
            group_start = (index // accumulation) * accumulation
            divisor = min(accumulation, len(loader) - group_start)
            loss = model(**{k: v.to(DEVICE) for k, v in batch.items()}).loss
            if not torch.isfinite(loss):
                raise RuntimeError("Loss NaN/Inf; audio/transcriptni tekshiring.")
            (loss / divisor).backward()
            if (index + 1) % accumulation == 0 or index + 1 == len(loader):
                torch.nn.utils.clip_grad_norm_([p for _, p in trainable], 1.0,
                                              error_if_nonfinite=True)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
        score = greedy_validation()
        history.append({"epoch": epoch + 1, **score})
        # WER kamayishi va CER yomonlashmasligi birgalikda talab qilinadi.
        if score["wer"] < best_metric["wer"] and score["cer"] <= best_metric["cer"]:
            best_metric, improved = score, True
            best_state = {name: p.detach().cpu().clone() for name, p in trainable}
        else:
            break
    model.load_state_dict(best_state, strict=False)
    model.eval()
    if improved:
        model_tag = "step8500_finetuned"
        model.save_pretrained(OUT / "finetuned_adapter", safe_serialization=True)
        processor.save_pretrained(OUT / "finetuned_adapter/processor")
    pd.DataFrame(history).to_csv(OUT / "acoustic_training_history.csv", index=False)

In [ ]:
corpus_path = OUT / "lm_train_only.txt"
train_sentences = sorted(set(train_df["text"]))
assert not (set(train_sentences) & heldout_texts)
corpus_path.write_text("\n".join(train_sentences) + "\n", encoding="utf-8")
existing_corpus = {normalize_text(line) for line in
                   (PROJECT / "kenlm/karakalpak_train.txt").read_text(encoding="utf-8").splitlines()
                   if line.strip()}
assert not (existing_corpus & heldout_texts), "Mavjud KenLM korpusida heldout gap bor."
assert existing_corpus <= set(train_df["text"]), "Mavjud LM tozalangan train bilan mos emas."
lm_paths = {"5gram_existing": PROJECT / "kenlm/karakalpak_5gram.binary"}
assert lm_paths["5gram_existing"].is_file()

if BUILD_EXTRA_LM:
    import subprocess
    tool_dir = PROJECT / "kenlm/kenlm_source/build/bin"
    assert (tool_dir / "lmplz").is_file() and (tool_dir / "build_binary").is_file()
    for order in (3, 5):
        arpa = OUT / f"train_only_{order}gram.arpa"
        binary = OUT / f"train_only_{order}gram.binary"
        with corpus_path.open("rb") as source, arpa.open("wb") as target:
            subprocess.run([str(tool_dir / "lmplz"), "-o", str(order),
                            "--discount_fallback", "-S", "25%", "-T", str(OUT)],
                           stdin=source, stdout=target, check=True)
        subprocess.run([str(tool_dir / "build_binary"), str(arpa), str(binary)], check=True)
        lm_paths[f"{order}gram_train_only"] = binary

In [ ]:
def model_fingerprint():
    digest = hashlib.sha256()
    digest.update(str(BASE_MODEL).encode())
    digest.update(json.dumps(model.config.to_dict(), sort_keys=True, default=str).encode())
    digest.update(json.dumps(model.peft_config["default"].to_dict(),
                             sort_keys=True, default=str).encode())
    digest.update(json.dumps(processor.feature_extractor.to_dict(),
                             sort_keys=True, default=str).encode())
    digest.update(json.dumps(vocab, sort_keys=True, ensure_ascii=False).encode())
    for name, parameter in sorted(model.named_parameters()):
        if parameter.requires_grad:
            digest.update(name.encode())
            digest.update(parameter.detach().cpu().float().contiguous().numpy().tobytes())
    return digest.hexdigest()

model_hash = model_fingerprint()

def make_cache(dataset, split):
    # Source metadata va shard size/mtime o‘zgarishi cache kalitini yangilaydi.
    digest = hashlib.sha256()
    digest.update(dataset.df[["id", "source_file", "source_row", "text"]].to_json().encode())
    for name in sorted(set(dataset.df["source_file"])):
        stat = (PROJECT / "audio_shards" / name).stat()
        digest.update(f"{name}:{stat.st_size}:{stat.st_mtime_ns}".encode())
    folder = OUT / "logits" / model_hash[:16] / f"{split}_{digest.hexdigest()[:16]}"
    folder.mkdir(parents=True, exist_ok=True)
    records = []
    for index in tqdm(range(len(dataset)), desc=f"{split} logits"):
        row = dataset.df.iloc[index]
        path = folder / f"sample_{index}.npy"
        if not path.exists():
            sample = dataset[index]
            np.save(path, get_logits(sample), allow_pickle=False)
        logits = np.load(path, allow_pickle=False)
        assert logits.ndim == 2 and logits.shape[1] == len(vocab)
        assert len(logits) and np.isfinite(logits).all()
        records.append({"id": row["id"], "reference": row["text"],
                        "logits_path": str(path.relative_to(PROJECT))})
    result = pd.DataFrame(records)
    result.to_csv(folder / "manifest.csv", index=False)
    return result

val_cache = make_cache(val_data, "val")
# Validation kichik; tuningda modelni qayta-qayta chaqirmaymiz.
val_logits = [np.load(PROJECT / path, allow_pickle=False)
              for path in val_cache["logits_path"]]
val_refs = val_cache["reference"].tolist()

In [ ]:
def metrics(refs, hypotheses):
    refs = [normalize_text(x) for x in refs]
    hypotheses = [normalize_text(x) for x in hypotheses]
    assert len(refs) == len(hypotheses) and len(refs)
    words = process_words(refs, hypotheses)
    return {"wer": wer(refs, hypotheses) * 100, "cer": cer(refs, hypotheses) * 100,
            "substitutions": words.substitutions, "deletions": words.deletions,
            "insertions": words.insertions, "samples": len(refs)}

decoders = {name: build_ctcdecoder(labels, kenlm_model_path=str(path))
            for name, path in lm_paths.items()}
plain_decoder = build_ctcdecoder(labels)
BASE_CONFIG = {"mode": "kenlm", "lm": "5gram_existing", "alpha": 0.65,
               "beta": 1.5, "beam_width": 200, "temperature": 1.0,
               "blank_bias": 0.0, "unk_score_offset": -10.0,
               "beam_prune_logp": -10.0, "token_min_logp": -5.0}

def decode_all(logits_list, config):
    if config["mode"] == "greedy":
        return [normalize_text(processor.decode(x.argmax(-1))) for x in logits_list]
    decoder = plain_decoder if config["mode"] == "beam" else decoders[config["lm"]]
    if config["mode"] == "kenlm":
        decoder.reset_params(alpha=config["alpha"], beta=config["beta"],
                             unk_score_offset=config["unk_score_offset"],
                             lm_score_boundary=True)
    predictions = []
    for raw in logits_list:
        logits = raw.copy() / config["temperature"]
        logits[:, blank_id] += config["blank_bias"]
        text = decoder.decode(logits, beam_width=int(config["beam_width"]),
                              beam_prune_logp=config["beam_prune_logp"],
                              token_min_logp=config["token_min_logp"])
        predictions.append(normalize_text(text))
    return predictions

baseline_predictions = decode_all(val_logits, BASE_CONFIG)
baseline_metrics = metrics(val_refs, baseline_predictions)
comparisons = []
for label, config in [("Greedy", {**BASE_CONFIG, "mode": "greedy"}),
                      ("Beam without LM", {**BASE_CONFIG, "mode": "beam"}),
                      ("Beam + existing KenLM", BASE_CONFIG)]:
    comparisons.append({"decoder": label, **metrics(val_refs, decode_all(val_logits, config))})
display(pd.DataFrame(comparisons))

In [ ]:
trials = []

def evaluate_config(config, stage):
    score = metrics(val_refs, decode_all(val_logits, config))
    trials.append({**config, **score, "stage": stage})
    pd.DataFrame(trials).to_csv(OUT / "validation_tuning.csv", index=False)
    return score

def choose_best():
    valid = [t for t in trials if t["wer"] <= baseline_metrics["wer"]
             and t["cer"] <= baseline_metrics["cer"]]
    winner = min(valid, key=lambda t: (t["wer"], t["cer"], t["beam_width"]))
    return {key: winner[key] for key in BASE_CONFIG}

# Baseline konfiguratsiya ham nomzodlar ichida.
trials.append({**BASE_CONFIG, **baseline_metrics, "stage": "baseline"})
for lm_name, alpha, beta in tqdm(list(product(
        lm_paths, [0.35, 0.5, 0.65, 0.8], [0.5, 1.0, 1.5])), desc="Alpha/beta"):
    evaluate_config({**BASE_CONFIG, "lm": lm_name, "alpha": alpha, "beta": beta}, "lm")
best_config = choose_best()

stage_base = best_config.copy()
for temperature, blank_bias in tqdm(list(product(
        [0.85, 1.0, 1.15], [-0.25, 0.0, 0.25])), desc="Acoustic calibration"):
    evaluate_config({**stage_base, "temperature": temperature,
                     "blank_bias": blank_bias}, "calibration")
best_config = choose_best()

stage_base = best_config.copy()
for beam_width, prune, token_min in [(100, -10.0, -5.0), (300, -10.0, -5.0),
                                    (200, -15.0, -7.0)]:
    evaluate_config({**stage_base, "beam_width": beam_width,
                     "beam_prune_logp": prune, "token_min_logp": token_min}, "beam")
best_config = choose_best()

stage_base = best_config.copy()
for offset in [-5.0, -15.0]:
    evaluate_config({**stage_base, "unk_score_offset": offset}, "unknown_word")
best_config = choose_best()
display(pd.DataFrame(trials).sort_values(["wer", "cer"]).head(10))

In [ ]:
best_val_predictions = decode_all(val_logits, best_config)
best_val_metrics = metrics(val_refs, best_val_predictions)
config_record = {
    "checkpoint": str(CHECKPOINT.relative_to(PROJECT)), "model_tag": model_tag,
    "model_fingerprint": model_hash,
    "lm_path": str(lm_paths[best_config["lm"]].relative_to(PROJECT)),
    "selection_split": "val", "normalization": "NFC + lowercase + whitespace",
    "baseline_validation": baseline_metrics, "selected_validation": best_val_metrics,
    "decoder": best_config
}
(OUT / "selected_decoder.json").write_text(
    json.dumps(config_record, ensure_ascii=False, indent=2), encoding="utf-8"
)
errors = val_cache[["id", "reference"]].copy()
errors["baseline_prediction"] = baseline_predictions
errors["optimized_prediction"] = best_val_predictions
errors["word_errors"] = [
    sum([r.substitutions, r.deletions, r.insertions])
    for r in (process_words(ref, hyp) for ref, hyp in zip(val_refs, best_val_predictions))
]
errors.to_csv(OUT / "validation_predictions.csv", index=False)
display(errors.sort_values("word_errors", ascending=False).head(10))
print("Baseline:", baseline_metrics)
print("Selected:", best_val_metrics)

In [ ]:
if RUN_FINAL_TEST:
    test_cache = make_cache(test_data, "test")
    test_logits = [np.load(PROJECT / path, allow_pickle=False)
                   for path in test_cache["logits_path"]]
    test_refs = test_cache["reference"].tolist()
    raw_refs = test_df["cleaned_text_lowercase"].tolist()
    predictions = test_cache[["id", "reference"]].copy()
    predictions["reference_raw"] = raw_refs
    test_scores = []
    configs = {"greedy": {**BASE_CONFIG, "mode": "greedy"},
               "beam_no_lm": {**BASE_CONFIG, "mode": "beam"},
               "baseline_kenlm": BASE_CONFIG, "optimized_kenlm": best_config}
    for name, config in configs.items():
        hypotheses = decode_all(test_logits, config)
        predictions[name] = hypotheses
        test_scores.append({"decoder": name, **metrics(test_refs, hypotheses),
                            "raw_reference_wer": wer(raw_refs, hypotheses) * 100,
                            "raw_reference_cer": cer(raw_refs, hypotheses) * 100})
    predictions.to_csv(OUT / "test_predictions.csv", index=False)
    score_table = pd.DataFrame(test_scores)
    score_table.to_csv(OUT / "test_metrics.csv", index=False)
    display(score_table)
    base_score = score_table.loc[score_table.decoder == "baseline_kenlm"].iloc[0]
    new_score = score_table.loc[score_table.decoder == "optimized_kenlm"].iloc[0]
    print("WER kamayishi (foiz punkt):", base_score.wer - new_score.wer)
    print("CER kamayishi (foiz punkt):", base_score.cer - new_score.cer)
else:
    print("Test bajarilmadi: RUN_FINAL_TEST=False.")